# Create BC-augmentation dataset

Converts the per-location BC-augmentation simulations (magnitude/timing perturbations, one
all-locations-desynchronized event, one catchment-emptying-from-baseline event, one genuinely
combined magnitude case) onto `template_100m.pkl`, then merges them with the existing q050-q150
uniform-scaling events into ONE training pkl:

    database/datasets/train/ahr_river_v03_marg_bcaugment_additionalsrc_velocity_100m_warmstart_multisim.pkl

Background: the standard `multisim` training only ever scales all 7 BC source locations together,
uniformly. The leave-one-out sensitivity ladder in `utils/visualize_bc_nonuniform_sensitivity.ipynb`
showed the resulting model's response to a single location changing doesn't track that location's
real volume share (see the `bc-sensitivity-uniform-training-gap` memory note) — these simulations
perturb one location (or a genuinely combined/desynchronized set) at a time, to teach the model to
distinguish each BC's individual effect.

`desync2` and `emptying_postpeak` are deliberately held out (converted here but NOT merged) — they
test *compositional* generalization on combinations never presented together during training.

Script twin for hal8: `run_convert_bc_augmentation.py` (same conversion+merge, via
`convert_sfincs_to_pkl_marg.py`'s `main()`). Same conversion code as `create_dataset_multisim.ipynb`.

In [5]:
import os, sys

# Resolve the repo root robustly (works in VS Code and nbconvert, any start cwd)
try:
    REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__vsc_ipynb_file__), '..'))
except NameError:
    _here = os.getcwd()
    REPO_ROOT = _here if os.path.isdir(os.path.join(_here, 'database')) \
                else os.path.abspath(os.path.join(_here, '..'))
assert os.path.isdir(os.path.join(REPO_ROOT, 'database')), f'Not the repo root: {REPO_ROOT}'
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print('Repo root:', REPO_ROOT)

Repo root: c:\Users\marrocol\OneDrive - Stichting Deltares\Documents\mSWE-GNN\mSWE-GNN_marg\mSWE-GNN_marg


## Config

In [6]:
TEMPLATE_PKL = 'database/datasets/train/template_100m.pkl'
SIM_ROOT     = 'database/raw_datasets_ahr/Simulations'
OUT_ROOT     = 'database/datasets'

WATER_LEVEL_VAR = 'zs'
BED_LEVEL_VAR   = 'zb'
VX_VAR = 'u'
VY_VAR = 'v'

TRAIN_SIMS = {
    # magnitude -- bidirectional (locations where the ladder showed the model is demonstrably broken)
    'kirmutscheid025x':      'ahr_river_v03_Marg_kirmutscheid025x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'kirmutscheid2x':        'ahr_river_v03_Marg_kirmutscheid2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'kreuzberg025x':         'ahr_river_v03_Marg_kreuzberg025x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'kreuzberg2x':           'ahr_river_v03_Marg_kreuzberg2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'niederadenau025x':      'ahr_river_v03_Marg_niederadenau025x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'niederadenau2x':        'ahr_river_v03_Marg_niederadenau2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'new2_025x':             'ahr_river_v03_Marg_new2_025x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'new2_2x':               'ahr_river_v03_Marg_new2_2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    # magnitude -- single direction (unremarkable ladder response, lower priority)
    'denn2x':                'ahr_river_v03_Marg_denn2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'muesch2x':              'ahr_river_v03_Marg_muesch2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'new1_2x':               'ahr_river_v03_Marg_new1_2x_additionalsrc_velocity_100m_cutpolygon_warmstart',
    # timing -- Kirmutscheid solo, +-1.5x its own rise time (8.0h -> 12h)
    'kirmutscheid_shiftm12h': 'ahr_river_v03_Marg_kirmutscheid_shiftm12h_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'kirmutscheid_shiftp12h': 'ahr_river_v03_Marg_kirmutscheid_shiftp12h_additionalsrc_velocity_100m_cutpolygon_warmstart',
    # all-locations-desynchronized (timing) and catchment-emptying-from-baseline
    'alldesync':             'ahr_river_v03_Marg_alldesync_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'alldrain':              'ahr_river_v03_Marg_alldrain_additionalsrc_velocity_100m_cutpolygon_warmstart',
    # genuinely combined magnitude perturbation (kept in training, not held out)
    'kirmutscheid15x_kreuzberg05x': 'ahr_river_v03_Marg_kirmutscheid15x_kreuzberg05x_additionalsrc_velocity_100m_cutpolygon_warmstart',
}

TEST_SIMS = {
    # held out on purpose -- compositional generalization checks, see module docstring
    'desync2':           'ahr_river_v03_Marg_desync2_additionalsrc_velocity_100m_cutpolygon_warmstart',
    'emptying_postpeak': 'ahr_river_v03_Marg_emptying_postpeak_additionalsrc_velocity_100m_cutpolygon_warmstart',
}

# Existing uniform-scaling events from run_convert_multisim.py / create_dataset_multisim.ipynb,
# already converted under a different naming pattern -- merged in below, not reconverted here.
EXISTING_TRAIN_TAGS = ['q050', 'q075', 'q125', 'q150']
EXISTING_PER_SIM_NAME = 'ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart_{tag}'

PER_SIM_NAME = 'ahr_river_v03_marg_{tag}_additionalsrc_velocity_100m_warmstart'
MERGED_NAME  = 'ahr_river_v03_marg_bcaugment_additionalsrc_velocity_100m_warmstart_multisim'

FORCE_REBUILD_PKL = False  # True to reconvert even if the per-scenario pkl already exists

for tag, folder in {**TRAIN_SIMS, **TEST_SIMS}.items():
    sim_dir = os.path.join(SIM_ROOT, folder)
    ok = all(os.path.exists(os.path.join(sim_dir, f)) for f in ['sfincs_map.nc', 'sfincs.src', 'sfincs.dis'])
    print(f"{tag:<30} {'OK' if ok else 'MISSING FILES'}  {sim_dir}")

kirmutscheid025x               OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_kirmutscheid025x_additionalsrc_velocity_100m_cutpolygon_warmstart
kirmutscheid2x                 OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_kirmutscheid2x_additionalsrc_velocity_100m_cutpolygon_warmstart
kreuzberg025x                  OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_kreuzberg025x_additionalsrc_velocity_100m_cutpolygon_warmstart
kreuzberg2x                    OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_kreuzberg2x_additionalsrc_velocity_100m_cutpolygon_warmstart
niederadenau025x               OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_niederadenau025x_additionalsrc_velocity_100m_cutpolygon_warmstart
niederadenau2x                 OK  database/raw_datasets_ahr/Simulations\ahr_river_v03_Marg_niederadenau2x_additionalsrc_velocity_100m_cutpolygon_warmstart
new2_025x                      OK  database/raw_datasets_ahr/Sim

## Step 1 — Template
Must already exist (`template_100m.pkl`, same one every other 4-scale dataset uses) — these events
have to live on the same mesh as the 1.0x dataset, so this notebook never rebuilds it.

In [7]:
assert os.path.exists(TEMPLATE_PKL), f'Template not found: {TEMPLATE_PKL} -- run build_template.py first'
print('Template found:', TEMPLATE_PKL)

Template found: database/datasets/train/template_100m.pkl


## Step 2 — Convert every training + held-out test simulation onto the template
(same code as `create_dataset_multisim.ipynb` Step 2, looped over `TRAIN_SIMS` + `TEST_SIMS`)

In [8]:
import pickle
import numpy as np
import xarray as xr

from database.convert_sfincs_to_pkl_marg import (
    load_single_data_object,
    get_target_points,
    get_source_points,
    interpolate_time_series,
    parse_src_file,
    parse_dis_file,
    build_output_data,
)

print('Loading template...')
template_data = load_single_data_object(TEMPLATE_PKL)
target_points = get_target_points(template_data)
print('  Template mesh faces:', target_points.shape[0])

for tag, folder in {**TRAIN_SIMS, **TEST_SIMS}.items():
    dataset_name = PER_SIM_NAME.format(tag=tag)
    sim_dir = os.path.join(SIM_ROOT, folder)
    out_train = os.path.join(OUT_ROOT, 'train', dataset_name + '.pkl')
    if not FORCE_REBUILD_PKL and os.path.exists(out_train):
        print('Skipping (already exists):', out_train)
        continue

    print()
    print('Processing:', tag, '->', dataset_name)
    map_path = os.path.join(sim_dir, 'sfincs_map.nc')
    ds = xr.open_dataset(map_path, decode_times=False)
    source_points = get_source_points(ds)

    zs = ds[WATER_LEVEL_VAR].values
    zb = ds[BED_LEVEL_VAR].values
    # SFINCS writes zs=NaN for DRY cells: fill with bed level so they enter as WD=0
    zs_filled = np.where(np.isnan(zs), zb[None, :, :], zs)
    WD_grid = np.maximum(zs_filled - zb[None, :, :], 0.0).astype(np.float32)
    print('  Interpolating WD...')
    WD = interpolate_time_series(source_points, WD_grid, target_points, 'WD')

    ds_raw = xr.open_dataset(map_path, decode_times=False, mask_and_scale=False)
    if VX_VAR and VX_VAR in ds.data_vars:
        VX_raw = ds_raw[VX_VAR].values.astype(np.float32)
        fv = ds_raw[VX_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VX_raw[VX_raw == fv] = np.nan
        print('  Interpolating VX...')
        VX = interpolate_time_series(source_points, VX_raw, target_points, 'VX')
    else:
        VX = np.zeros_like(WD)
    if VY_VAR and VY_VAR in ds.data_vars:
        VY_raw = ds_raw[VY_VAR].values.astype(np.float32)
        fv = ds_raw[VY_VAR].attrs.get('_FillValue', None)
        if fv is not None:
            VY_raw[VY_raw == fv] = np.nan
        print('  Interpolating VY...')
        VY = interpolate_time_series(source_points, VY_raw, target_points, 'VY')
    else:
        VY = np.zeros_like(WD)
    ds_raw.close()

    time_var = ds.coords.get('time', ds.coords.get('t', None))
    map_times_s = (time_var.values.astype(np.float64) if time_var is not None
                   else np.arange(zs.shape[0]) * 3600.0)
    ds.close()

    print('  Reading src/dis files...')
    src_xy = parse_src_file(os.path.join(sim_dir, 'sfincs.src'))
    dis_times_s, discharge = parse_dis_file(os.path.join(sim_dir, 'sfincs.dis'))
    print(' ', len(src_xy), 'source points, discharge shape:', discharge.shape)

    data_out = build_output_data(
        template_data, WD=WD, VX=VX, VY=VY,
        map_times_s=map_times_s, src_xy=src_xy,
        dis_times_s=dis_times_s, discharge=discharge,
    )

    for split in ['train', 'test']:
        os.makedirs(os.path.join(OUT_ROOT, split), exist_ok=True)
        with open(os.path.join(OUT_ROOT, split, dataset_name + '.pkl'), 'wb') as f:
            pickle.dump([data_out], f)
    print('  Saved train+test:', dataset_name + '.pkl')
    print('  WD=', tuple(data_out.WD.shape),
          '| node_BC=', data_out.node_BC.tolist(),
          '| Q peak =', float(data_out.BC[:, :, 1].max()), 'm3/s')

Loading template...
  Template mesh faces: 30079
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_kirmutscheid025x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_kirmutscheid2x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_kreuzberg025x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_kreuzberg2x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_niederadenau025x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_niederadenau2x_additionalsrc_velocity_100m_warmstart.pkl
Skipping (already exists): database/datasets\train\ahr_river_v03_marg_new2_025x_additionalsrc_velocity_100m_warmstart.pkl

Processing: new2_2x -> ahr_river_v03_marg_new2_2x_additionalsrc_ve

## Step 3 — Merge `TRAIN_SIMS` + the existing q050-q150 events into one training pkl
`TEST_SIMS` (`desync2`, `emptying_postpeak`) stay as separate individual test pkls — evaluated one
at a time in `visualize_bc_nonuniform_sensitivity.ipynb`, deliberately NOT merged here.

In [9]:
merged = []
for tag in TRAIN_SIMS:
    pkl_path = os.path.join(OUT_ROOT, 'train', PER_SIM_NAME.format(tag=tag) + '.pkl')
    with open(pkl_path, 'rb') as f:
        data_list = pickle.load(f)
    for data in data_list:
        print(f'{tag}: node_BC = {data.node_BC.tolist()}   WD peak = {float(data.WD.max()):.3f} m   '
              f'n_steps = {data.WD.shape[1]}')
    merged += data_list

for tag in EXISTING_TRAIN_TAGS:
    pkl_path = os.path.join(OUT_ROOT, 'train', EXISTING_PER_SIM_NAME.format(tag=tag) + '.pkl')
    assert os.path.exists(pkl_path), \
        f'{tag}: expected pre-existing pkl not found at {pkl_path} -- run create_dataset_multisim.ipynb first'
    with open(pkl_path, 'rb') as f:
        data_list = pickle.load(f)
    for data in data_list:
        print(f'{tag} (existing): node_BC = {data.node_BC.tolist()}   WD peak = {float(data.WD.max()):.3f} m   '
              f'n_steps = {data.WD.shape[1]}')
    merged += data_list

out_path = os.path.join(OUT_ROOT, 'train', MERGED_NAME + '.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(merged, f)
print(f'\nSaved {len(merged)} training events '
      f'({len(TRAIN_SIMS)} BC-augmentation + {len(EXISTING_TRAIN_TAGS)} existing uniform-scaling) -> {out_path}')
print('Train with config_best_sweep_bcaugment.yaml '
      '(validation/test = the held-out 1.0x event; desync2/emptying_postpeak stay independent).')

kirmutscheid025x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 9.652 m   n_steps = 121
kirmutscheid2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 15.336 m   n_steps = 121
kreuzberg025x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 10.473 m   n_steps = 121
kreuzberg2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 11.044 m   n_steps = 121
niederadenau025x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 10.401 m   n_steps = 121
niederadenau2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 11.548 m   n_steps = 121
new2_025x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 10.465 m   n_steps = 121
new2_2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 11.026 m   n_steps = 121
denn2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 11.012 m   n_steps = 121
muesch2x: node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]   WD peak = 11.286 m   n

## Step 4 — Sanity check the held-out `TEST_SIMS`

Data-integrity checks only (not model evaluation — that's `visualize_bc_nonuniform_sensitivity.ipynb`'s
job). Confirms `node_BC` matches the reference 1.0x event for both, and specifically for
`emptying_postpeak`, confirms the converted `.pkl` actually shows a draining trend (domain-wide WDsum
decreasing over the ~450-step rollout) before anyone trains or tests against it.

In [ ]:
REF_1X_PKL = 'database/datasets/train/ahr_river_v03_marg_additionalsrc_velocity_100m_warmstart.pkl'
with open(REF_1X_PKL, 'rb') as f:
    ref_1x = pickle.load(f)
node_bc_ref = ref_1x[0].node_BC.tolist()
print(f'1.0x reference node_BC = {node_bc_ref}\n')

for tag in TEST_SIMS:
    pkl_path = os.path.join(OUT_ROOT, 'test', PER_SIM_NAME.format(tag=tag) + '.pkl')
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)[0]
    same_bc = data.node_BC.tolist() == node_bc_ref
    print(f'{tag}: node_BC identical to reference: {same_bc}   n_steps = {data.WD.shape[1]}')
    assert same_bc, f'{tag}: node_BC differs from the 1.0x event!'

    if tag == 'emptying_postpeak':
        node_ptr = data.node_ptr.numpy()
        wd_s0 = data.WD[node_ptr[0]:node_ptr[1]].clamp(min=0).numpy()
        ws = wd_s0.sum(0)
        print(f'  WDsum: t=0 {ws[0]:.0f}  t=-1 {ws[-1]:.0f}  '
              f'({ws[-1] / ws[0] * 100:.1f}% of initial remains)')
        assert ws[-1] < ws[0], \
            'emptying_postpeak does not show a draining trend -- check the raw SFINCS run before using it as a test set'
        print('  Draining trend confirmed (WDsum decreases over the rollout).')

1.0x reference node_BC = [12555, 7287, 15, 368, 5453, 3676, 6427]

desync2: node_BC identical to reference: True   n_steps = 121
emptying_postpeak: node_BC identical to reference: True   n_steps = 451
  WDsum: t=0 7766  t=-1 136  (1.8% of initial remains)
  Draining trend confirmed (WDsum decreases over the rollout).


: 